# MOZYME GPU: perfil por secciones (CPU vs GPU-DIAGG vs SCF residente)

Clona `https://github.com/juvenalyosa/mopac_gpu`, compila MOPAC con CUDA y corre `scripts/mozyme_section_profile.py` sobre las moléculas de publicación.

Runtime: **GPU (A100 o H100 preferido; T4/L4 funcionan pero con FP64 lento)**.

Abrir directamente desde GitHub: `https://colab.research.google.com/github/juvenalyosa/mopac_gpu/blob/main/colab/mozyme_diagg_profile_colab.ipynb`


## 1. GPU

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)

## 2. Obtener el código fuente (git clone; opcionalmente un zip local)

In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile

REPO_URL = 'https://github.com/juvenalyosa/mopac_gpu.git'
BRANCH = 'main'
USE_LOCAL_ZIP = False  # True: subir un zip creado con scripts/create_colab_gpu_zip.py en vez de clonar

CONTENT = Path('/content')
SRC = CONTENT / 'mopac_src'
BUILD = CONTENT / 'mopac_build'

if SRC.exists():
    shutil.rmtree(SRC)
if USE_LOCAL_ZIP:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    SRC.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(SRC)
    entries = [p for p in SRC.iterdir() if not p.name.startswith('.')]
    if len(entries) == 1 and entries[0].is_dir() and (entries[0] / 'CMakeLists.txt').exists():
        SRC = entries[0]
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(SRC)], check=True)
    print(subprocess.run(['git', '-C', str(SRC), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert (SRC / 'CMakeLists.txt').exists(), f'CMakeLists.txt not found under {SRC}'
assert (SRC / 'scripts' / 'mozyme_section_profile.py').exists(), 'scripts/mozyme_section_profile.py missing'
print('source dir:', SRC)

## 3. Compilar MOPAC con GPU

In [ ]:
import subprocess, shutil

def run(cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    return subprocess.run([str(c) for c in cmd], check=True, **kw)

# apt-get needs fresh package lists in a new Colab session (exit status 100 otherwise);
# retry once because the Ubuntu mirrors occasionally time out.
pkgs = ['cmake', 'gfortran', 'ninja-build', 'libblas-dev', 'liblapack-dev']
for attempt in range(2):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    r = subprocess.run(['apt-get', 'install', '-y', '-qq', '--no-install-recommends', *pkgs],
                       check=False, capture_output=True, text=True)
    if r.returncode == 0:
        break
    print(r.stdout[-2000:], r.stderr[-2000:])
    if attempt == 1:
        raise SystemExit('apt-get install failed twice; run !apt-get update && !apt-get install ' + ' '.join(pkgs))
if BUILD.exists():
    shutil.rmtree(BUILD)
cmake_cmd = ['cmake', '-S', SRC, '-B', BUILD, '-GNinja', '-DGPU=ON', '-DTESTS=OFF', '-DCMAKE_BUILD_TYPE=RelWithDebInfo']
try:
    run(cmake_cmd + ['-DCUDA_ARCHS=native'])
except subprocess.CalledProcessError:
    shutil.rmtree(BUILD, ignore_errors=True)
    run(cmake_cmd + ['-DCUDA_ARCHS=all'])
run(['cmake', '--build', BUILD, '--target', 'mopac', '--parallel', '2'])
MOPAC = BUILD / 'mopac'
assert MOPAC.exists(), 'mopac executable not built'
print('OK:', MOPAC)

## 4. Perfil por secciones

Modos: `cpu` (referencia), `gpu-diagg` (loop SCF en CPU con sólo DIAGG en los kernels paralelos nuevos), `resident` (SCF completo residente en GPU, modo estricto). Empieza con crambina (~30 s por modo en CPU); ubiquitina tarda ~40 s por modo.

## 4b. Depuración de los kernels DIAGG (crambina, modo gpu-diagg)

Corre sólo crambina en `gpu-diagg` con `MOPAC_MOZYME_DIAGG_DEBUG=1` (sincroniza tras cada kernel e imprime `[DIAGG DEBUG] ...`), y luego bajo `compute-sanitizer` (memcheck) para detectar accesos inválidos en device y bajo `--tool racecheck` para carreras en memoria compartida. Muestra las últimas líneas de cada corrida.

In [ ]:
import os, shutil, subprocess
RUN_RACECHECK = False  # True: add the compute-sanitizer racecheck pass (slow)

DBG = CONTENT / 'diagg_debug'
shutil.rmtree(DBG, ignore_errors=True)
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop'

def run_debug(tag, prefix, mode='gpu-diagg', head=0, deck=deck, debug='1'):
    env = dict(os.environ, MOPAC_MOZYME_DIAGG_DEBUG=debug)
    cmd = [*prefix, sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', mode, '--out-dir', str(DBG / tag), '--timeout', '1800']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True, env=env)
    print(proc.stdout[-6000:])
    log = next(iter((DBG / tag).rglob('combined.log')), None)
    if log:
        lines = log.read_text(errors='ignore').splitlines()
        import re
        pat = re.compile(r'DIAGG DEBUG|GPU ERROR|Backtrace|\.cu:\d|\.F90:\d|^=+|Invalid|Error|error|FINAL HEAT|MOZYME GPU diagg|CYCLE:|resident_publish|resident_upload|MOZYME GPU SCF\] status')
        keep = [l for l in lines if pat.search(l)]
        if head:
            print('\n'.join(keep[:head]))
            print('   ...')
        print('\n'.join(keep[-60:]))
        (CONTENT / f'diagg_debug_{tag}.log').write_text('\n'.join(lines))

run_debug('plain', [])
run_debug('resident_debug', [], mode='resident', head=60)
# Hand-back regression (1SCF, ITRY=70): DENOUT=5 makes the resident SCF hand back after 4 iterations;
# the CPU then finishes that SCF from the published device state (the driver blocks the GPU for the
# rest of the SCF).  Expect one "status=fallback_cpu", no [GPU ERROR], heat -2901.681 within 0.01.
# A diverging CPU continuation shows as ITRY exhausted (~70 iterations, wrong or missing heat).
run_debug('denout_handback', [], mode='resident', debug='0',
          deck=SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_denout.mop')
san = shutil.which('compute-sanitizer') or '/usr/local/cuda/bin/compute-sanitizer'
if Path(san).exists():
    run_debug('sanitizer', [san, '--target-processes', 'all', '--print-limit', '20'])
    # racecheck: shared-memory races (diagg2 block kernel shares its staged lists across 8 warps).
    # 10-50x slower than memcheck (10-20 min on crambin): off by default, enable after touching shared memory.
    if RUN_RACECHECK:
        run_debug('racecheck', [san, '--tool', 'racecheck', '--target-processes', 'all', '--print-limit', '20'])
else:
    print('compute-sanitizer not found; skipping')


## 4c. Optimización de geometría (crambina, 3 ciclos): CPU vs residente

Mide pasos de optimización completos (SCF warm-start + gradientes). Los gradientes MOZYME hoy corren en CPU (`dcart_gradient` en la tabla).

El modo `resident-gradcheck` evalúa el gradiente MOZYME en CPU y en GPU en cada ciclo y escribe `[MOZYME GPU gradient] check ... max_abs_diff= rms_diff=` (se conserva el resultado CPU); `resident` usa directamente el gradiente GPU.


In [ ]:
import subprocess
opt_deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_opt.mop'
OPT_OUT = CONTENT / 'mozyme_opt_profile'
cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(opt_deck),
       '--modes', 'cpu,resident-gradcheck,resident', '--out-dir', str(OPT_OUT), '--timeout', '3600']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
import re
pat = re.compile(r'MOZYME tidy|MOZYME GPU SCF\] status|MOZYME GPU SCF\] resident_(upload|publish)|MOZYME GPU gradient|MOZYME GPU hcore|MOZYME GPU disp|strict_abort|CYCLE:|HEAT OF FORMATION|GRADIENT NORM|GPU ERROR|Backtrace|\.F90:\d')
for log in sorted(OPT_OUT.rglob('combined.log')):
    print('=====', log.relative_to(OPT_OUT))
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
    print('\n'.join(lines[:60]))


In [ ]:
import os, subprocess

MOLECULES = [
    'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop',
    'benchmarks/publication_inputs/mop/protein_ubiquitin_1ubq.mop',
]
MODES = 'cpu,resident,default'
OUT = CONTENT / 'mozyme_section_profile'

cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC),
       *[str(SRC / m) for m in MOLECULES], '--modes', MODES, '--out-dir', str(OUT), '--timeout', '3600']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
(CONTENT / 'mozyme_section_profile_summary.txt').write_text(proc.stdout + '\n' + proc.stderr)

## 4c-full. Optimización de crambina hasta convergencia: CPU vs resident

Hasta ahora todas las optimizaciones fueron de 3 ciclos sin converger, y el heat del ciclo 3 varía ~0.8 kcal/mol por el
orden no determinista de diagg2. Aquí se deja converger (hasta 100 ciclos, GNORM por defecto): los heats finales deben
coincidir dentro de 0.05 kcal/mol y la tabla del último paso da el coste real por paso caliente. CPU ~10 min.


In [ ]:
import subprocess, re
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_optfull.mop'
OUT_F = CONTENT / 'mozyme_optfull_profile'
cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(deck),
       '--modes', 'cpu,resident', '--out-dir', str(OUT_F), '--timeout', '7200']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
out = proc.stdout.splitlines()
print('\n'.join(l for l in out if 'wall=' in l))
start = next((i for i, l in enumerate(out) if l.startswith('last geometry step')), None)
if start is not None:
    print('\n'.join(out[start:start + 30]))
pat = re.compile(r'CYCLE:|HEAT OF FORMATION|GRADIENT NORM|GPU ERROR|fallback_cpu reason|strict_abort')
for log in sorted(OUT_F.rglob('combined.log')):
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
    cycles = [l for l in lines if 'CYCLE:' in l]
    print('=====', log.relative_to(OUT_F), f'({len(cycles)} cycles)')
    print('\n'.join(cycles[:3] + ['   ...'] + cycles[-3:] + [l for l in lines if 'CYCLE:' not in l][-6:]))


## 4e. Hand-back: el SCF en CPU tras devolver el control (regresión)

Con `DENOUT=5` el SCF residente devuelve el control a la CPU tras 4 iteraciones. Diagnóstico del 18-09-2026: con los
ayudantes GPU antiguos del bucle CPU (diagg1 aocc/avir, density batch, eimp, cnvgz, helecz, rotprep) la CPU no convergía
(ITRY agotado) y con solo `diagg1_avir` apagado convergía a -2899.57 en vez de -2901.68. Desde entonces esos ayudantes son
opt-in. Aquí la corrida por defecto debe converger a -2901.68 (dentro de 0.01) en ~10 s; `legacy_helpers_on` reproduce el fallo
(~2 min) y `cpu_reference` da la referencia.


In [ ]:
import subprocess, re
DIAG = CONTENT / 'handback_diag'
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_denout.mop'
VARIANTS = [
    # (tag, mode, overrides)  -- overrides applied after the mode; KEY= unsets
    ('default_handback',    'resident',  ['MOPAC_GPU_VERBOSE=1']),
    ('legacy_helpers_on',   'resident',  ['MOPAC_GPU_VERBOSE=1', 'MOPAC_MOZYME_DIAGG1_AOCC_GPU=1',
                                          'MOPAC_MOZYME_DIAGG1_AVIR_GPU=1', 'MOPAC_MOZYME_DENSITY_BATCH_GPU=1',
                                          'MOPAC_MOZYME_EIMP_GPU=1', 'MOPAC_MOZYME_CNVGZ_GPU=1',
                                          'MOPAC_MOZYME_HELECZ_GPU=1', 'MOPAC_MOZYME_DIAGG2_ROTPREP_GPU=1']),
    ('cpu_reference',       'cpu',       []),
]
pat = re.compile(r'MOZYME GPU SCF\] status|MOZYME GPU SCF\] resident_(upload|publish)|GPU ERROR|FINAL HEAT|ITERATIONS|SCF CALCULATION FAILED|strict_abort')
for tag, mode, sets in VARIANTS:
    cmd = [sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', mode, '--out-dir', str(DIAG), '--timeout', '900', '--tag', '_' + tag]
    for item in sets:
        cmd += ['--set', item]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    out = proc.stdout.splitlines()
    summary = [l for l in out if 'wall=' in l or 'dHf' in l or 'fallback_cpu=' in l]
    print('\n'.join(summary))
    if tag == 'legacy_helpers_on':   # section table: which CPU-loop section is slow (diagg 1.8 s/call vs 0.5 on CPU)
        start = next((i for i, l in enumerate(out) if l.startswith('section')), None)
        if start is not None:
            print('\n'.join(out[start:start + 22]))
    for log in sorted((DIAG / deck.stem / (mode + '_' + tag)).rglob('combined.log')):
        keep = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
        print('\n'.join(keep[-12:]))
    print()


## 4d. Sistemas grandes: DNA (1BNA), tRNA (1EHZ), adenilato quinasa (1AKE, ~7000 átomos) y 1AKE sin ligando (apo, 6689 átomos)

Single points en modo `resident` y `default` (y `cpu` sólo para 1BNA/1EHZ; para 1AKE la CPU tarda ~1 h — active `RUN_CPU_1AKE` si quiere la referencia). Los calores de referencia CPU quedan en el `.out` de la corrida `cpu`.


In [ ]:
import subprocess
RUN_CPU_1AKE = False     # CPU reference for 1AKE apo (~10 min on Colab CPU; Mac reference: -30137.44797 kcal/mol)
RUN_FULL_1AKE = False    # 1AKE with the AP5 ligand: does not converge on CPU either (~70 min); off by default
RUN_TRNA = False         # tRNA 1EHZ: does not converge on CPU either (~20 min per mode); off by default
LARGE = [
    ('benchmarks/publication_inputs/mop/dna_dodecamer_1bna.mop', 'cpu,resident,default'),
    # protein only (the AP5 ligand carries alternate conformations that hurt SCF convergence on CPU and GPU)
    ('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake_apo.mop',
     'cpu,resident,resident-noindex,default' if RUN_CPU_1AKE else 'resident,resident-noindex,default'),
]
if RUN_TRNA:
    LARGE.append(('benchmarks/publication_inputs/mop/rna_trna_1ehz.mop', 'cpu,resident,default'))
if RUN_FULL_1AKE:
    LARGE.append(('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake.mop', 'resident,default'))
# 3-cycle geometry optimization of 1AKE apo: the "last geometry step" table is the warm-start
# per-step cost (SCF from the previous LMOs + hcore + gradient) that optimization/MD pays.
LARGE.append(('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake_apo_opt.mop', 'resident'))
OUT_L = CONTENT / 'mozyme_large_profile'
for deck, modes in LARGE:
    cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(SRC / deck),
           '--modes', modes, '--out-dir', str(OUT_L), '--timeout', '7200']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    print(proc.stdout)
    print(proc.stderr[-3000:])
    # host<->device transfer profile (bytes, ms, GB/s) and SCF status of each run of this deck
    import re
    pat = re.compile(r'MOZYME tidy|MOZYME GPU SCF\] resident_(upload|publish)|MOZYME GPU SCF\] status|GPU ERROR|CYCLE:')
    for log in sorted((OUT_L / Path(deck).stem).rglob('combined.log')):
        keep = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
        print('=====', log.relative_to(OUT_L))
        print('\n'.join(keep[-24:]))
(CONTENT / 'mozyme_large_profile_summary.txt').write_text('done')


## 5. Marcadores GPU y descarga

Muestra las líneas `[MOZYME GPU ...]` relevantes de cada corrida (éxito, fallback, abort) y empaqueta los logs para descargar.

In [ ]:
import re, shutil
from google.colab import files

pattern = re.compile(r'\[MOZYME GPU (diagg1_construct|diagg2_rotate|SCF)\]|strict_abort|fallback_cpu|MOZYME_RESIDENT_STAGE|GPU ERROR|CUDA error|cuda_context|Fortran runtime error|Segmentation|Program received|free\(\)|malloc|Backtrace|FINAL HEAT|SCF CALCULATION FAILED')
for log in sorted(OUT.rglob('combined.log')):
    print('=====', log.relative_to(OUT))
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pattern.search(l)]
    for l in lines[:40]:
        print('  ', l[:200])
    if len(lines) > 40:
        print(f'   ... {len(lines) - 40} more')

archive = shutil.make_archive(str(CONTENT / 'mozyme_section_profile_logs'), 'zip', OUT)
files.download(archive)
files.download(str(CONTENT / 'mozyme_section_profile_summary.txt'))